# fairness-method-combinations-liver-disease 

## Kombinationseffekte ausgewählter Pre- und Post-Processing-Fairnessmethoden bei der Lebererkrankungsvorhersage

## Imports

In [ ]:
from pathlib import Path
import logging
import time

import numpy as np
import pandas as pd
from scipy.stats import t as student_t

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

# AIF360 meldet beim Import ungenutzte Zusatzmodule (z. B. TensorFlow).
# Nur diese Meldungen werden unterdrueckt, Laufzeitwarnungen bleiben sichtbar.
_previous_logging_disable_level = logging.root.manager.disable
logging.disable(logging.WARNING)
try:
    from aif360.datasets import BinaryLabelDataset
    from aif360.algorithms.preprocessing import Reweighing as AIF360Reweighing
    from aif360.algorithms.postprocessing import EqOddsPostprocessing
finally:
    logging.disable(_previous_logging_disable_level)

from fairlearn.preprocessing import CorrelationRemover
from fairlearn.postprocessing import ThresholdOptimizer

# Fehlt eine der vier Kernbibliotheken, bricht der Import mit klarer Fehlermeldung ab.
AIF360_RW_AVAILABLE = True
AIF360_EO_AVAILABLE = True
FAIRLEARN_CORR_AVAILABLE = True
FAIRLEARN_THRESHOLD_AVAILABLE = True

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

## Reproduzierbarkeit: benötigte Requirements


In [ ]:
import sys
from importlib.metadata import version, PackageNotFoundError

REQUIRED_PYTHON = "3.12.4"
REQUIRED_PACKAGES = [
    "pandas==2.2.3",
    "numpy==2.2.5",
    "scipy==1.15.3",
    "scikit-learn==1.6.1",
    "aif360==0.6.1",
    "fairlearn==0.12.0",
    "openpyxl==3.1.5",
]

_warnings = []

# Python-Version pruefen (nur MAJOR.MINOR.MICRO exakt, wie bei den Paketen).
actual_python = f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"
if actual_python != REQUIRED_PYTHON:
    _warnings.append(f"Python: erwartet {REQUIRED_PYTHON}, gefunden {actual_python}")

# Installierte Paketversionen gegen die gepinnten Anforderungen pruefen.
for requirement in REQUIRED_PACKAGES:
    pkg_name, _, expected_version = requirement.partition("==")
    try:
        installed_version = version(pkg_name)
    except PackageNotFoundError:
        _warnings.append(f"{pkg_name}: nicht installiert (erwartet {expected_version})")
        continue
    if installed_version != expected_version:
        _warnings.append(f"{pkg_name}: erwartet {expected_version}, gefunden {installed_version}")

if _warnings:
    warning_text = "\n".join(_warnings)
    raise RuntimeError(
        "Umgebung weicht von den in REQUIRED_PACKAGES gepinnten Versionen ab:\n"
        f"{warning_text}\n\n"
        "Fuer reproduzierbare Ergebnisse bitte exakt diese Versionen installieren, "
        "z. B. via requirements.txt, oder REQUIRED_PYTHON/REQUIRED_PACKAGES bewusst anpassen."
    )

print(f"Python=={actual_python}  (OK, entspricht REQUIRED_PYTHON)")
for requirement in REQUIRED_PACKAGES:
    print(f"{requirement}  (OK)")

## Konfiguration

In [ ]:
# Zentrale Konfiguration aller Experiment-Parameter an einer Stelle.

# Findet den Projektordner (das Github-Repo-Verzeichnis, in dem "data" liegt),
# unabhaengig davon, aus welchem Unterordner (z.B. "notebook/") Jupyter
# gestartet wurde: es wird ab dem aktuellen Verzeichnis so lange nach oben
# gegangen, bis ein Ordner mit einem "data"-Unterordner gefunden wird.
def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir():
            return candidate
    print(
        "WARNUNG: Kein 'data'-Ordner in der Verzeichnis-Hierarchie gefunden. "
        f"Verwende Fallback-Projektordner: {start}"
    )
    return start

_CURRENT_DIR = Path.cwd()
PROJECT_ROOT = _find_project_root(_CURRENT_DIR)
print(f"Projektordner erkannt: {PROJECT_ROOT.resolve()}")

DATA_DIR = PROJECT_ROOT / "data" if (PROJECT_ROOT / "data").is_dir() else PROJECT_ROOT
RESULTS_DIR = PROJECT_ROOT / "results"

# Anzahl Wiederholungen pro Methode (wie bei Straw et al. 2022), um
# Zufallsschwankungen einzelner Splits auszugleichen.
N_REPEATS = 100

# Testanteil (70/30-Split), identisch zu Straw et al.
TEST_SIZE = 0.30

# Basis-Seed; Lauf i verwendet BASE_SEED + i fuer reproduzierbare, aber
# unterschiedliche Splits und RF-Initialisierungen.
BASE_SEED = 42

# Anteil des Trainingssets, der fuer Post-Processing (Equalized Odds,
# ThresholdOptimizer) als eigenes Kalibrierungsset abgezweigt wird - trennt
# Training und Kalibrierung, um Data Leakage zu vermeiden.
VALIDATION_SIZE_POSTPROCESSING = 0.25

# Staerke der Korrelationsentfernung: alpha=1.0 = vollstaendig, alpha=0.0 = keine Aenderung.
CORRELATION_REMOVER_ALPHA = 1.0

# Hyperparameter des Random-Forest-Basismodells (bei Straw et al. leistungsstark,
# aber auch stark von Disparitaeten betroffen).
RF_PARAMS = {
    "n_estimators": 200,  # Anzahl der Baeume im Wald
    "n_jobs": -1,         # nutzt alle CPU-Kerne, um die Laufzeit bei 100x9x2 Laeufen zu begrenzen
}

# Dateinamen der beiden Eingabedatensaetze. Beide Dateien muessen im DATA_DIR liegen.
DATASET_FILES = {
    "ILPD":     "Indian Liver Patient Dataset (ILPD).csv",
    "HCV Data": "hcvdat0.csv",
}

# Datensatzspezifische Konfiguration: ILPD und HCV nutzen unterschiedliche
# Spaltennamen, daher Kandidatenlisten - die erste vorhandene Spalte wird jeweils verwendet.
DATASET_CONFIG = {
    "ILPD": {
        "target_candidates":  ["selector", "dataset", "is_patient"],  # moegliche Namen der Zielspalte
        "sex_candidates":     ["gender", "sex"],                      # moegliche Namen der Geschlechtsspalte
        "target_description": "1 = Lebererkrankung, 0 = keine Lebererkrankung",
    },
    "HCV Data": {
        "target_candidates":  ["category"],
        "sex_candidates":     ["sex", "gender"],
        "target_description": "1 = Hepatitis/Fibrosis/Cirrhosis, 0 = Blood Donor bzw. suspect Blood Donor",
    },
}

# Prueft je Datensatz, ob die Konfiguration vollstaendig ist.
def _is_dataset_configured(dataset_name: str) -> bool:
    cfg = DATASET_CONFIG.get(dataset_name, {})
    required_keys = {"target_candidates", "sex_candidates", "target_description"}
    return (
        bool(DATASET_FILES.get(dataset_name))
        and required_keys.issubset(cfg)
        and bool(cfg["target_candidates"])
        and bool(cfg["sex_candidates"])
    )

for dataset_name in DATASET_FILES:
    if _is_dataset_configured(dataset_name):
        print(f"{dataset_name}: erfolgreich konfiguriert")
    else:
        print(f"{dataset_name}: nicht erfolgreich konfiguriert")
        raise ValueError(f"Unvollständige Konfiguration für {dataset_name}.")

## Datensaetze laden

In [ ]:
# ILPD-Kopien liegen mit oder ohne Kopfzeile vor; ohne Kopfzeile wird die
# UCI-Standardreihenfolge der Spalten verwendet.
ILPD_STANDARD_COLUMNS = [
    "age", "gender", "total_bilirubin", "direct_bilirubin", "alkphos",
    "sgpt", "sgot", "total_proteins", "albumin", "a_g_ratio", "selector"
]


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Vereinheitlicht Spaltennamen (Gross-/Kleinschreibung, Leerzeichen, Sonderzeichen)."""
    df = df.copy()
    df.columns = (
        pd.Index(df.columns)
        .astype(str)
        .str.strip()                              # fuehrende/abschliessende Leerzeichen entfernen
        .str.lower()                               # alles auf Kleinbuchstaben
        .str.replace(r"\s+", "_", regex=True)      # Leerzeichen(-folgen) durch Unterstrich ersetzen
        .str.replace(r"[^a-z0-9_]", "", regex=True)  # alles, was kein a-z/0-9/_ ist, entfernen (z.B. Klammern, %)
    )
    # Entfernt eine evtl. vorhandene Index-Spalte ("Unnamed: 0") aus CSV-Exporten.
    return df.loc[:, ~df.columns.str.startswith("unnamed")]


def load_ilpd(path: Path) -> pd.DataFrame:
    """Laedt das ILPD robust gegenueber beiden Dateiformaten (mit/ohne Kopfzeile)."""
    # Schritt 1: Versuch, die Datei MIT automatisch erkannter Kopfzeile zu lesen.
    try:
        df = standardize_columns(pd.read_csv(path))
        # Erfolgreich, wenn Zielspalte "selector" und eine Geschlechtsspalte vorhanden sind.
        if "selector" in df.columns and ("gender" in df.columns or "sex" in df.columns):
            return df
    except Exception:
        # Einlesen schlaegt z.B. fehl, wenn die erste Zeile schon Zahlenwerte
        # enthaelt - dann direkt zum Fallback in Schritt 2.
        pass
    # Schritt 2 (Fallback): ohne Kopfzeile einlesen, Standardspaltennamen manuell zuweisen.
    return standardize_columns(pd.read_csv(path, header=None, names=ILPD_STANDARD_COLUMNS))


def load_hcv(path: Path) -> pd.DataFrame:
    """Der HCV-Datensatz liegt durchgehend mit Standardkopfzeile vor, daher reicht ein einfacher Read."""
    return standardize_columns(pd.read_csv(path))


def load_dataset(name: str) -> pd.DataFrame:
    """Dispatcher-Funktion: waehlt anhand des Datensatznamens die passende Ladefunktion aus."""
    path = DATA_DIR / DATASET_FILES[name]
    if not path.exists():
        # Klare Fehlermeldung, falls die Datei nicht existiert.
        raise FileNotFoundError(f"Datei nicht gefunden: {path}")
    if name == "ILPD":
        return load_ilpd(path)
    if name == "HCV Data":
        return load_hcv(path)
    raise KeyError(f"Unbekannter Datensatz: {name}")


# Alle konfigurierten Datensaetze laden. Ausgegeben wird nur der Status.
loaded_datasets = {}

for dataset_name in DATASET_FILES:
    try:
        loaded_datasets[dataset_name] = load_dataset(dataset_name)
    except Exception as exc:
        print(f"{dataset_name}: Laden fehlgeschlagen")
        raise RuntimeError(f"{dataset_name} konnte nicht geladen werden.") from exc
    print(f"{dataset_name}: erfolgreich geladen")

## Hilfsfunktionen

In [ ]:
def resolve_column_name(df: pd.DataFrame, candidates: list[str]) -> str | None:
    """Gibt den ersten in df vorhandenen Spaltennamen aus candidates zurueck (None, falls keiner existiert)."""
    cols = set(df.columns)
    for c in candidates:
        if c in cols:
            return c
    return None


def map_sex_values(series: pd.Series) -> pd.Series:
    """Vereinheitlicht Geschlechtscodierungen (z.B. "M"/"F", 1/0) zu "Maenner"/"Frauen";
    Unbekanntes wird als "Unbekannt" markiert."""
    mapping = {
        "m": "Maenner", "male": "Maenner", "mann": "Maenner",
        "maenner": "Maenner", "maennlich": "Maenner", "1": "Maenner",
        "f": "Frauen",  "female": "Frauen",  "frau": "Frauen",
        "frauen": "Frauen",  "weiblich": "Frauen",  "0": "Frauen",
    }
    # Normalisiert Zahl- oder Text-Eingaben (unabhaengig von Gross-/Kleinschreibung)
    # fuer ein einheitliches Mapping.
    return series.astype(str).str.strip().str.lower().map(mapping).fillna("Unbekannt")


def normalize_target_values(series: pd.Series, dataset_name: str) -> pd.Series:
    """Wandelt die Zielvariable in ein einheitliches Binaerformat um: 1 = erkrankt, 0 = gesund."""
    if dataset_name == "ILPD":
        # ILPD codiert im Original 1=krank, 2=gesund; wird hier auf 1/0 umkodiert.
        numeric = pd.to_numeric(series, errors="coerce")
        vals = set(numeric.dropna().unique())
        if vals.issubset({1, 2}):
            return numeric.map({1: 1, 2: 0})
        # Liegt bereits ein anderes Format vor, unveraendert zurueckgeben.
        return numeric

    if dataset_name == "HCV Data":
        # HCV nutzt Textlabels wie "0=Blood Donor", "1=Hepatitis" usw.;
        # Stadien 1-3 -> 1 (erkrankt), Blood-Donor-Varianten -> 0 (gesund).
        labels = series.astype(str).str.strip()
        low    = labels.str.lower()
        neg    = {"0=blood donor", "0s=suspect blood donor", "0", "0s"}
        pos    = {"1=hepatitis", "2=fibrosis", "3=cirrhosis", "1", "2", "3"}
        mapped = low.map(lambda x: 0 if x in neg else (1 if x in pos else np.nan))
        # Fallback: ist die Spalte bereits numerisch 0/1, diese Version verwenden.
        numeric = pd.to_numeric(labels, errors="coerce")
        if mapped.isna().all() and set(numeric.dropna().unique()).issubset({0, 1}):
            return numeric
        return mapped

    # Fallback fuer weitere Datensaetze: einfache numerische Konvertierung.
    return pd.to_numeric(series, errors="coerce")


def safe_div(n, d):
    """Division, die bei Nenner 0 np.nan statt eines Fehlers zurueckgibt."""
    return n / d if d else np.nan


def confusion_stats(y_true, y_pred) -> dict:
    """Berechnet die Konfusionsmatrix (TN, FP, FN, TP) und die daraus abgeleiteten Standardmetriken."""
    # labels=[0, 1] fixiert die Reihenfolge, auch wenn in der Stichprobe nur eine Klasse vorkommt.
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
        "Accuracy":       safe_div(tp + tn, tp + tn + fp + fn),  # Anteil korrekter Vorhersagen
        "TPR":            safe_div(tp, tp + fn),  # Sensitivitaet/Recall: Anteil korrekt erkannter Erkrankter
        "FPR":            safe_div(fp, fp + tn),  # 1 - Spezifitaet: Anteil faelschlich als krank Eingestufter
        "TNR":            safe_div(tn, tn + fp),  # Spezifitaet: Anteil korrekt erkannter Gesunder
        "FNR":            safe_div(fn, fn + tp),  # zentrale Metrik bei Straw et al.: Anteil UEBERSEHENER Erkrankter
        "Selection_Rate": safe_div((pd.Series(y_pred) == 1).sum(), len(y_pred)),  # Anteil positiver Vorhersagen insgesamt
        "PPV":            safe_div(tp, tp + fp),  # Praezision: wie viele der positiv Vorhergesagten sind tatsaechlich krank
    }


def evaluate_fairness_metrics(y_true, y_pred, sensitive_feature) -> dict:
    """
    Berechnet drei Fairnessmetriken als Differenz Frauen minus Maenner
    (negativ = Frauen schlechter):
    - SPD: Differenz der Rate positiver Vorhersagen.
    - AOD: Mittelwert aus FPR- und TPR-Differenz (Equalized-Odds-Kriterium).
    - PPD: Differenz der Praezision (PPV).
    """
    # Fuehrt Labels, Vorhersagen und Gruppe zusammen, um nach Geschlecht filtern zu koennen.
    df = pd.DataFrame({
        "y_true": y_true,
        "y_pred": y_pred,
        "group":  sensitive_feature,
    }).dropna()

    men   = df[df["group"] == "Maenner"]
    women = df[df["group"] == "Frauen"]

    # Fehlt eine Gruppe in der Testmenge, ist keine Differenz berechenbar.
    if men.empty or women.empty:
        return {"SPD": np.nan, "AOD": np.nan, "PPD": np.nan}

    m_rates = confusion_stats(men["y_true"],   men["y_pred"])
    w_rates = confusion_stats(women["y_true"], women["y_pred"])

    return {
        "SPD": w_rates["Selection_Rate"] - m_rates["Selection_Rate"],
        "AOD": 0.5 * (
            (w_rates["FPR"] - m_rates["FPR"]) +
            (w_rates["TPR"] - m_rates["TPR"])
        ),
        "PPD": w_rates["PPV"] - m_rates["PPV"],
    }


def make_joint_stratify_labels(y, sex) -> pd.Series:
    """Kombiniert Zielklasse und Geschlecht zu einem Schluessel (z.B. "1__Maenner")
    fuer die Stratifikation beim Split."""
    y_s   = pd.Series(y).reset_index(drop=True).astype(str)
    sex_s = map_sex_values(pd.Series(sex).reset_index(drop=True)).astype(str)
    return y_s + "__" + sex_s

## Datensatzvorbereitung

In [ ]:
def prepare_dataset_for_modeling(dataset_name: str) -> dict:
    """
    Fuehrt alle Vorbereitungsschritte vor dem Train/Test-Split durch (Spalten
    identifizieren, Zielvariable normalisieren, Geschlecht kodieren). Imputation
    und Skalierung erfolgen bewusst erst in der Pipeline nach dem Split, um
    Data Leakage zu vermeiden.
    """
    if dataset_name not in loaded_datasets:
        return {"success": False, "message": f"{dataset_name} ist nicht geladen."}

    df  = loaded_datasets[dataset_name].copy()
    cfg = DATASET_CONFIG[dataset_name]

    # Schritt 1: Ziel- und Geschlechtsspalte anhand der Kandidatenlisten ermitteln.
    target_col = resolve_column_name(df, cfg["target_candidates"])
    sex_col    = resolve_column_name(df, cfg["sex_candidates"])

    if target_col is None or sex_col is None:
        return {"success": False, "message": "Ziel- oder Geschlechtsspalte nicht gefunden."}

    # Schritt 2: Zielvariable auf 0/1 normalisieren, Geschlecht auf "Maenner"/"Frauen".
    y         = pd.to_numeric(normalize_target_values(df[target_col], dataset_name), errors="coerce")
    sex_group = map_sex_values(df[sex_col])

    # Schritt 3: Zeilen mit fehlender Zielvariable oder unklarem Geschlecht verwerfen.
    keep      = y.notna() & sex_group.isin(["Frauen", "Maenner"])
    df        = df.loc[keep].copy()
    y         = y.loc[keep].astype(int)
    sex_group = sex_group.loc[keep]

    # Sicherheitscheck: Zielvariable muss nach der Normalisierung ausschliesslich 0/1 enthalten.
    if set(pd.Series(y).dropna().unique()) != {0, 1}:
        return {"success": False, "message": "Zielvariable ist nach Normalisierung nicht binaer 0/1."}

    # Schritt 4: Feature-Matrix X aufbauen; Zielspalte entfernen, Geschlecht numerisch
    # kodieren (0=Frauen, 1=Maenner) und als Feature belassen (wie bei Straw et al.).
    X = df.drop(columns=[target_col]).copy()
    X[sex_col] = sex_group.map({"Frauen": 0.0, "Maenner": 1.0})

    # Schritt 5: alle Spalten numerisch konvertieren; nicht-numerische Werte werden
    # zu NaN (spaeter in der Pipeline imputiert).
    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")

    num_cols = X.columns.tolist()

    # Schritt 6: Preprocessing-Pipeline definieren (Imputation + Skalierung auf [0,1]);
    # gefittet wird erst spaeter, ausschliesslich auf den Trainingsdaten.
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="mean")),
                ("scaler",  MinMaxScaler()),
            ]), num_cols),
        ],
        remainder="drop"  # Spalten, die nicht in num_cols stehen (sollte es keine geben), werden verworfen
    )

    # Rueckgabe: Dictionary mit allem, was spaetere Schritte fuer diesen Datensatz benoetigen.
    return {
        "success":      True,
        "dataset_name": dataset_name,
        "config":       cfg,
        "target_col":   target_col,
        "sex_col":      sex_col,
        "X":            X,
        "y":            y,
        "sex_raw":      sex_group,
        "num_cols":     num_cols,
        "preprocessor": preprocessor,
    }


# Vorbereitung fuer jeden Datensatz. Ausgegeben wird nur der Status.
prepared_datasets = {}

for dataset_name in DATASET_FILES:
    prepared = prepare_dataset_for_modeling(dataset_name)
    prepared_datasets[dataset_name] = prepared

    if prepared["success"]:
        print(f"{dataset_name}: erfolgreich vorbereitet")
    else:
        print(f"{dataset_name}: Vorbereitung fehlgeschlagen")
        raise RuntimeError(prepared["message"])

## Experiment-Engine

In [ ]:
# Trennt die generische Ablauflogik (Splitten, Trainieren, Auswerten) von der
# Modelllogik: run_experiment() ruft dazu einen model_builder-Callback auf, der
# fuer jede Fairnessmethode unterschiedlich sein kann.

def split_prepared_data(prepared: dict, seed: int):
    """Fuehrt den 70/30 Train/Test-Split ohne Stratifikation durch (wie bei Straw et al.,
    zur Vergleichbarkeit mit der Referenzstudie)."""
    return train_test_split(
        prepared["X"],
        prepared["y"],
        prepared["sex_raw"],
        test_size=TEST_SIZE,
        random_state=seed,
    )


def build_random_forest_pipeline(preprocessor, seed: int) -> Pipeline:
    """
    Baut eine Pipeline aus Preprocessing (Imputation + Skalierung) und
    Random-Forest-Klassifikator; verhindert Data Leakage, da das Preprocessing
    nur auf den Trainingsdaten gefittet wird. clone() liefert dafuer stets eine
    frische, ungefittete Kopie.
    """
    return Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier",   RandomForestClassifier(random_state=seed, **RF_PARAMS)),
    ])


def run_experiment(prepared: dict, model_builder, model_label: str) -> dict:
    """
    Zentrale Experimentschleife: N_REPEATS Laeufe mit je neuem Split, Training,
    Vorhersage und Auswertung. model_builder ist ein Callback
    (prepared, seed, X_train, y_train, sex_train, X_test, sex_test) -> (model, y_pred),
    der die methodenspezifische Logik enthaelt; diese Funktion kennt nur
    Splitting, Schleife und Metrikberechnung.
    """
    if prepared is None or not prepared.get("success", False):
        raise ValueError("Keine gueltig vorbereiteten Daten uebergeben.")

    group_rows    = []  # sammelt pro Lauf UND pro Geschlechtsgruppe je eine Zeile mit Metriken
    fairness_rows = []  # sammelt pro Lauf eine Zeile mit den gruppenvergleichenden Fairnessmetriken

    for run_idx in range(N_REPEATS):
        # Jeder Lauf erhaelt einen eigenen, reproduzierbaren Seed fuer Split und
        # Random-Forest-Initialisierung.
        seed = BASE_SEED + run_idx

        X_train, X_test, y_train, y_test, sex_train, sex_test = split_prepared_data(prepared, seed)

        # Training und Vorhersage werden an den Callback delegiert - hier
        # unterscheiden sich die Methoden voneinander.
        _, y_pred = model_builder(
            prepared=prepared,
            seed=seed,
            X_train=X_train,
            y_train=y_train,
            sex_train=sex_train,
            X_test=X_test,
            sex_test=sex_test,
        )

        # Vorhersagen, wahre Labels und Geschlecht der Testmenge in einem DataFrame
        # zusammenfuehren, um gruppenweise auswerten zu koennen.
        run_df = pd.DataFrame({
            "true_label": pd.Series(y_test).reset_index(drop=True),
            "pred_label": pd.Series(y_pred).reset_index(drop=True),
            "Geschlecht": map_sex_values(pd.Series(sex_test).reset_index(drop=True)),
        })

        # Konfusionsmatrix-Metriken je Geschlechtsgruppe berechnen und als eigene
        # Zeile speichern.
        for group, gdf in run_df.groupby("Geschlecht"):
            s = confusion_stats(gdf["true_label"], gdf["pred_label"])
            group_rows.append({
                "Methode":      model_label,
                "Wiederholung": run_idx + 1,
                "Seed":         seed,
                "Geschlecht":   group,
                "Accuracy":     s["Accuracy"],
                "TPR":          s["TPR"],
                "FPR":          s["FPR"],
                "TNR":          s["TNR"],
                "FNR":          s["FNR"],
            })

        # Zusaetzlich die gruppenvergleichenden Fairnessmetriken (SPD/AOD/PPD) und die
        # Gesamt-Accuracy fuer diesen Lauf berechnen (eine Zeile pro Lauf).
        overall_stats = confusion_stats(run_df["true_label"], run_df["pred_label"])
        fairness = evaluate_fairness_metrics(
            run_df["true_label"],
            run_df["pred_label"],
            run_df["Geschlecht"],
        )
        fairness_rows.append({
            "Methode":      model_label,
            "Wiederholung": run_idx + 1,
            "Seed":         seed,
            "Accuracy":     overall_stats["Accuracy"],
            **fairness,
        })

    # Beide Zeilenlisten als DataFrames zurueckgeben - Rohdaten fuer die spaetere Auswertung.
    return {
        "group_runs_df":   pd.DataFrame(group_rows),
        "fairness_runs_df": pd.DataFrame(fairness_rows),
    }

## Pre-Processing-Methoden

In [ ]:
class SampleWeightPipelineWrapper(BaseEstimator, ClassifierMixin):
    """
    Sklearn-Pipelines erwarten sample_weight ueber den Schluessel
    "classifier__sample_weight" statt als normalen fit()-Parameter. Dieser
    Wrapper nimmt sample_weight regulaer entgegen und reicht ihn korrekt weiter.
    """

    def __init__(self, pipeline, fit_weight_param="classifier__sample_weight"):
        self.pipeline         = pipeline
        self.fit_weight_param = fit_weight_param

    def fit(self, X, y, sample_weight=None):
        # clone() erzeugt eine frische, ungefittete Kopie fuer jeden Lauf.
        self.pipeline_ = clone(self.pipeline)
        fit_kwargs = {}
        if sample_weight is not None:
            fit_kwargs[self.fit_weight_param] = sample_weight
        self.pipeline_.fit(X, y, **fit_kwargs)
        # classes_ wird von sklearn an mehreren Stellen erwartet (z.B. bei predict_proba).
        self.classes_ = getattr(self.pipeline_, "classes_", np.array([0, 1]))
        return self

    def predict(self, X):
        return self.pipeline_.predict(X)

    def predict_proba(self, X):
        return self.pipeline_.predict_proba(X)


class AIF360ReweighingClassifier(BaseEstimator, ClassifierMixin):
    """
    Reweighing nach Kamiran & Calders (2012) als sklearn-kompatibler Klassifikator:
    berechnet je Kombination aus Geschlecht und Zielklasse ein Korrekturgewicht,
    das statistische Unabhaengigkeit von Geschlecht und Zielvariable annaehert.
    Die Gewichte werden von AIF360 berechnet, der nachgelagerte Klassifikator
    wird damit trainiert.
    """

    def __init__(self, estimator, sex_series_train_reference):
        # estimator: Klassifikator, der mit den Reweighing-Gewichten trainiert wird
        # sex_series_train_reference: Geschlechts-Serie des Trainingssets, indiziert
        #   wie der urspruengliche DataFrame (fuer Zuordnung ueber X.index)
        self.estimator                  = estimator
        self.sex_series_train_reference = sex_series_train_reference

    def _sex_binary(self, X_index) -> pd.Series:
        """Liest fuer die uebergebenen Zeilenindizes das Geschlecht aus und kodiert es binaer (Frauen=0, Maenner=1)."""
        groups     = map_sex_values(self.sex_series_train_reference.loc[X_index])
        sex_binary = groups.map({"Frauen": 0.0, "Maenner": 1.0})
        # Bei verbleibendem "Unbekannt" oder NaN: Abbruch statt stillschweigend falscher Gewichte.
        if sex_binary.isna().any() or (groups == "Unbekannt").any():
            raise ValueError("AIF360 Reweighing unterstuetzt nur klar zuordenbare Maenner/Frauen-Werte.")
        return sex_binary.astype(float)

    def _binary_label_dataset(self, X: pd.DataFrame, y) -> "BinaryLabelDataset":
        """Konvertiert X (nur der Index wird benoetigt) und y in das von AIF360
        vorausgesetzte BinaryLabelDataset-Format."""
        df = pd.DataFrame({
            "label":         pd.Series(y, index=X.index).astype(float),
            "sex_protected": self._sex_binary(X.index).to_numpy(),
        }, index=X.index)

        return BinaryLabelDataset(
            favorable_label=1.0,    # die "positive" Klasse aus AIF360-Sicht ist hier "krank" = 1
            unfavorable_label=0.0,
            df=df,
            label_names=["label"],
            protected_attribute_names=["sex_protected"],
            privileged_protected_attributes=[[1.0]],    # Maenner = privilegierte Gruppe
            unprivileged_protected_attributes=[[0.0]],  # Frauen  = nicht-privilegierte Gruppe
        )

    def fit(self, X, y):
        if not AIF360_RW_AVAILABLE:
            raise ImportError("AIF360 Reweighing ist nicht verfuegbar.")

        # Schritt 1: AIF360-Datenformat erzeugen, Reweighing-Gewichte berechnen lassen.
        train_ds  = self._binary_label_dataset(X, y)
        reweigher = AIF360Reweighing(
            privileged_groups=[{"sex_protected": 1.0}],
            unprivileged_groups=[{"sex_protected": 0.0}],
        )
        train_ds_rw   = reweigher.fit_transform(train_ds)
        # instance_weights enthaelt fuer jede Trainingszeile das berechnete Gewicht.
        sample_weight = np.asarray(train_ds_rw.instance_weights).ravel()

        # Schritt 2: Klassifikator mit diesen Gewichten trainieren.
        self.estimator_ = clone(self.estimator)
        self.estimator_.fit(X, y, sample_weight=sample_weight)
        self.classes_ = getattr(self.estimator_, "classes_", np.array([0, 1]))
        return self

    def predict(self, X):
        return self.estimator_.predict(X)

    def predict_proba(self, X):
        return self.estimator_.predict_proba(X)


def build_correlation_remover_pipeline(prepared: dict, seed: int) -> Pipeline:
    """
    Baut eine Pipeline aus Preprocessing, CorrelationRemover und Klassifikator.
    Der CorrelationRemover entfernt die lineare Korrelation der Features zum
    Geschlecht (alpha=1.0 = vollstaendig) und veraendert damit - anders als
    Reweighing - direkt den Feature-Raum statt des Trainings.
    """
    if not FAIRLEARN_CORR_AVAILABLE:
        raise ImportError("Fairlearn CorrelationRemover ist nicht verfuegbar.")
    if prepared["sex_col"] not in prepared["num_cols"]:
        raise ValueError("Geschlechtsspalte ist nicht in den numerischen Features enthalten.")

    # CorrelationRemover erwartet das sensitive Feature als numerischen Index
    # (nicht Spaltenname), ermittelt aus num_cols.
    sensitive_idx = prepared["num_cols"].index(prepared["sex_col"])

    return Pipeline([
        ("preprocessor",        clone(prepared["preprocessor"])),  # Imputation + Skalierung
        ("correlation_remover", CorrelationRemover(
            sensitive_feature_ids=[sensitive_idx],
            alpha=CORRELATION_REMOVER_ALPHA,
        )),
        ("classifier",          RandomForestClassifier(random_state=seed, **RF_PARAMS)),
    ])

## Modellaufbau und Methodenkombinationen

In [ ]:
def make_base_or_pre_estimator(prepared: dict, seed: int,
                               X_train, sex_train, pre: str | None):
    """
    Gibt je nach pre den passenden, noch ungefitteten Schaetzer zurueck:
    None -> Baseline-Pipeline, "reweighing" -> AIF360ReweighingClassifier,
    "correlation_remover" -> Pipeline mit CorrelationRemover-Schritt.
    """
    base_pipeline = build_random_forest_pipeline(prepared["preprocessor"], seed)

    if pre is None:
        return base_pipeline

    if pre == "reweighing":
        # Pipeline wird in SampleWeightPipelineWrapper gepackt, damit sample_weight
        # korrekt durchgereicht wird.
        return AIF360ReweighingClassifier(
            estimator=SampleWeightPipelineWrapper(base_pipeline),
            sex_series_train_reference=pd.Series(sex_train, index=X_train.index),
        )

    if pre == "correlation_remover":
        return build_correlation_remover_pipeline(prepared, seed)

    raise ValueError(f"Unbekanntes Preprocessing: {pre}")


def split_for_postprocessing(X_train, y_train, sex_train, seed: int):
    """
    Teilt das Trainingsset erneut: 75% fuer das Modelltraining, 25% als
    Kalibrierungsset fuer die Post-Processing-Methoden. Stratifiziert nach
    Klasse x Geschlecht; bei zu kleinen Teilgruppen ohne Stratifikation.
    """
    try:
        return train_test_split(
            X_train, y_train, sex_train,
            test_size=VALIDATION_SIZE_POSTPROCESSING,
            random_state=seed,
            stratify=make_joint_stratify_labels(y_train, sex_train),
        )
    except ValueError:
        # Fallback ohne Stratifikation bei zu kleinen Gruppen.
        return train_test_split(
            X_train, y_train, sex_train,
            test_size=VALIDATION_SIZE_POSTPROCESSING,
            random_state=seed,
        )


def make_binary_dataset(y, sex, scores=None, y_pred=None) -> "BinaryLabelDataset":
    """Konvertiert Labels/Vorhersagen und Geschlecht ins AIF360-BinaryLabelDataset-
    Format (fuer apply_eqodds_postprocessing)."""
    groups     = map_sex_values(pd.Series(sex).reset_index(drop=True))
    sex_binary = groups.map({"Frauen": 0.0, "Maenner": 1.0})

    # Je nach Aufrufkontext werden entweder die wahren Labels (y) oder die
    # Modellvorhersagen (y_pred) als "label"-Spalte verwendet.
    labels = y if y_pred is None else y_pred
    df = pd.DataFrame({
        "label":         pd.Series(labels).reset_index(drop=True).astype(float),
        "sex_protected": sex_binary.astype(float),
    })
    if scores is not None:
        # Scores werden nur fuer das Vorhersage-Dataset benoetigt.
        df["score"] = pd.Series(scores).reset_index(drop=True).astype(float)

    return BinaryLabelDataset(
        favorable_label=1.0,
        unfavorable_label=0.0,
        df=df,
        label_names=["label"],
        scores_names=["score"] if "score" in df.columns else [],
        protected_attribute_names=["sex_protected"],
        privileged_protected_attributes=[[1.0]],
        unprivileged_protected_attributes=[[0.0]],
    )


def apply_eqodds_postprocessing(base_model, X_train, y_train, sex_train,
                                X_test, sex_test, seed: int):
    """
    Wendet AIF360 Equalized Odds Postprocessing an: 1) Split in Fit- (75%) und
    Kalibrierungsset (25%). 2) Basismodell auf dem Fit-Set trainieren.
    3) Postprocessor auf dem Kalibrierungsset fitten, um TPR/FPR zwischen den
    Geschlechtern anzugleichen. 4) Testset-Vorhersagen ueber den Postprocessor korrigieren.
    """
    if not AIF360_EO_AVAILABLE:
        raise ImportError("AIF360 Equalized Odds ist nicht verfuegbar.")

    X_fit, X_val, y_fit, y_val, sex_fit, sex_val = split_for_postprocessing(
        X_train, y_train, sex_train, seed
    )

    # Schritt 2: Basismodell trainieren (clone() liefert eine frische Kopie).
    fitted_model = clone(base_model)
    fitted_model.fit(X_fit, y_fit)

    # Schritt 3a: Vorhersagen auf dem Kalibrierungsset (Score = Wahrscheinlichkeit
    # fuer Klasse 1, Pred = binarisiert bei Schwelle 0.5).
    val_scores = fitted_model.predict_proba(X_val)[:, 1]
    val_pred   = (val_scores >= 0.5).astype(int)

    # true_ds: wahre Labels, pred_ds: Vorhersagen - beide fuer den Postprocessor benoetigt.
    true_ds = make_binary_dataset(y_val, sex_val)
    pred_ds = make_binary_dataset(y_val, sex_val, scores=val_scores, y_pred=val_pred)

    # Schritt 3b: den Postprocessor auf Basis dieser beiden Datasets fitten.
    eq = EqOddsPostprocessing(
        privileged_groups=[{"sex_protected": 1.0}],
        unprivileged_groups=[{"sex_protected": 0.0}],
        seed=seed,
    )
    eq.fit(true_ds, pred_ds)

    # Schritt 4: Testset-Vorhersagen erzeugen und durch den Postprocessor korrigieren.
    test_scores  = fitted_model.predict_proba(X_test)[:, 1]
    test_pred    = (test_scores >= 0.5).astype(int)
    test_pred_ds = make_binary_dataset(
        np.zeros(len(test_pred)), sex_test, scores=test_scores, y_pred=test_pred
    )
    fair_pred_ds = eq.predict(test_pred_ds)

    return fitted_model, fair_pred_ds.labels.ravel().astype(int)


def apply_threshold_optimizer_postprocessing(base_model, X_train, y_train, sex_train,
                                             X_test, sex_test, seed: int):
    """
    Wendet den Fairlearn ThresholdOptimizer (Equalized-Odds-Nebenbedingung) an,
    analog zu apply_eqodds_postprocessing(): Split, Training auf dem Fit-Set,
    Fitten gruppenspezifischer Schwellenwerte auf dem Kalibrierungsset, dann
    Vorhersage auf dem Testset. Die try/except-Bloecke fangen ab, dass
    random_state je Fairlearn-Version an unterschiedlicher Stelle erwartet wird.
    """
    if not FAIRLEARN_THRESHOLD_AVAILABLE:
        raise ImportError("Fairlearn ThresholdOptimizer ist nicht verfuegbar.")

    X_fit, X_val, y_fit, y_val, sex_fit, sex_val = split_for_postprocessing(
        X_train, y_train, sex_train, seed
    )

    fitted_model = clone(base_model)
    fitted_model.fit(X_fit, y_fit)

    # ThresholdOptimizer erwartet sensitive_features als String-Labels, daher erneut map_sex_values.
    groups_val  = map_sex_values(pd.Series(sex_val).reset_index(drop=True))
    groups_test = map_sex_values(pd.Series(sex_test).reset_index(drop=True))

    optimizer_kwargs = {
        "estimator":      fitted_model,
        "constraints":    "equalized_odds",   # Fairnessnebenbedingung: TPR und FPR angleichen
        "objective":      "accuracy_score",   # innerhalb der Nebenbedingung Accuracy maximieren
        "predict_method": "predict_proba",    # Optimizer arbeitet auf Wahrscheinlichkeiten, nicht auf harten Labels
        "prefit":         True,               # fitted_model ist bereits trainiert, soll NICHT erneut gefittet werden
    }

    try:
        optimizer = ThresholdOptimizer(**optimizer_kwargs, random_state=seed)
    except TypeError:
        optimizer = ThresholdOptimizer(**optimizer_kwargs)

    # Schritt 3: ThresholdOptimizer auf dem KALIBRIERUNGSset fitten.
    optimizer.fit(X_val, y_val, sensitive_features=groups_val)

    # Schritt 4: Vorhersagen fuer das eigentliche TESTset erzeugen.
    try:
        y_pred = optimizer.predict(X_test, sensitive_features=groups_test, random_state=seed)
    except TypeError:
        y_pred = optimizer.predict(X_test, sensitive_features=groups_test)

    return fitted_model, np.asarray(y_pred).astype(int)


def run_named_experiment(prepared_data: dict, model_label: str,
                         pre: str | None = None, post: str | None = None) -> dict:
    """
    Baut aus pre (Pre-Processing) und post (Post-Processing) einen Callback und
    fuehrt run_experiment() damit aus. pre=post=None ergibt die Baseline; sind
    beide gesetzt, entsteht eine der vier Kombinationsmethoden.
    """
    def builder(prepared, seed, X_train, y_train, sex_train, X_test, sex_test):
        # Schritt 1: Basis- bzw. Pre-Processing-Modell erzeugen (noch ungefittet).
        model = make_base_or_pre_estimator(prepared, seed, X_train, sex_train, pre=pre)

        # Schritt 2: je nach post-Parameter unterschiedlich trainieren/vorhersagen.
        if post is None:
            # Kein Post-Processing: auf dem gesamten Trainingsset trainieren,
            # direkt vorhersagen.
            model.fit(X_train, y_train)
            return model, model.predict(X_test)
        if post == "eqodds":
            return apply_eqodds_postprocessing(
                model, X_train, y_train, sex_train, X_test, sex_test, seed
            )
        if post == "threshold_optimizer":
            return apply_threshold_optimizer_postprocessing(
                model, X_train, y_train, sex_train, X_test, sex_test, seed
            )
        raise ValueError(f"Unbekanntes Postprocessing: {post}")

    return run_experiment(prepared_data, builder, model_label)


# -----------------------------------------------------------------------------
# METHODENMATRIX: alle 9 zu testenden Kombinationen
# -----------------------------------------------------------------------------
# 1 Baseline, 2 isolierte Pre-Processing-Methoden, 2 isolierte
# Post-Processing-Methoden, 4 Kombinationen aus je einer Pre- und
# Post-Processing-Methode. Die isolierten Methoden dienen spaeter als
# Referenz fuer den Vergleich mit den Kombinationen.
METHOD_SPECS = [
    # Baseline: reiner Random Forest ohne jede Fairnessintervention
    {"label": "Baseline-Random-Forest",                                                      "pre": None,                  "post": None},

    # Pre-Processing-Methoden isoliert (ohne Post-Processing)
    {"label": "AIF360 Reweighing + Random-Forest",                                           "pre": "reweighing",          "post": None},
    {"label": "Fairlearn CorrelationRemover + Random-Forest",                                "pre": "correlation_remover", "post": None},

    # Post-Processing-Methoden isoliert (ohne Pre-Processing)
    {"label": "AIF360 Equalized Odds + Random-Forest",                                       "pre": None,                  "post": "eqodds"},
    {"label": "Fairlearn ThresholdOptimizer + Random-Forest",                                "pre": None,                  "post": "threshold_optimizer"},

    # Kombination 1: Reweighing (Pre) + Equalized Odds (Post)
    {"label": "AIF360 Reweighing + AIF360 Equalized Odds + Random-Forest",                  "pre": "reweighing",          "post": "eqodds"},

    # Kombination 2: Reweighing (Pre) + ThresholdOptimizer (Post)
    {"label": "AIF360 Reweighing + Fairlearn ThresholdOptimizer + Random-Forest",            "pre": "reweighing",          "post": "threshold_optimizer"},

    # Kombination 3: CorrelationRemover (Pre) + Equalized Odds (Post)
    {"label": "Fairlearn CorrelationRemover + AIF360 Equalized Odds + Random-Forest",        "pre": "correlation_remover", "post": "eqodds"},

    # Kombination 4: CorrelationRemover (Pre) + ThresholdOptimizer (Post)
    {"label": "Fairlearn CorrelationRemover + Fairlearn ThresholdOptimizer + Random-Forest", "pre": "correlation_remover", "post": "threshold_optimizer"},
]

print(f"Methodenmatrix: erfolgreich konfiguriert ({len(METHOD_SPECS)} Methoden)")

## Experimente ausführen

In [ ]:
# Fuehrt run_named_experiment() fuer jede Kombination aus METHOD_SPECS und jeden
# Datensatz aus; Ergebnisse landen in all_dataset_results.
# Laufzeit: bei 100 Wiederholungen x 9 Methoden x 2 Datensaetzen ca. 20-60 Min.,
# da je Lauf ein neuer Random Forest trainiert wird.

all_dataset_results = {}  # Struktur: {Datensatzname: {Methodenlabel: Ergebnis-Dict aus run_experiment}}

total_start = time.perf_counter()

for dataset_name, prepared in prepared_datasets.items():
    print("=" * 100)
    print(f"Datensatz: {dataset_name}")
    print("=" * 100)

    # Bricht ab, falls die Vorbereitung fuer diesen Datensatz fehlgeschlagen ist
    # (z.B. Datei fehlte).
    if not prepared.get("success", False):
        raise RuntimeError(
            f"{dataset_name} ist nicht vollständig vorbereitet: "
            f"{prepared.get('message', 'unbekannter Fehler')}"
        )

    dataset_results = {}

    for spec in METHOD_SPECS:
        label        = spec["label"]
        method_start = time.perf_counter()  # Laufzeitmessung dieser einzelnen Methode

        try:
            # Prueft, ob die fuer diese Methode benoetigte Bibliothek verfuegbar ist
            # (siehe *_AVAILABLE-Flags); fehlt sie, bricht der Lauf mit Fehler ab.
            if spec["pre"]  == "reweighing"          and not AIF360_RW_AVAILABLE:
                raise RuntimeError("AIF360 Reweighing ist nicht verfügbar.")
            if spec["pre"]  == "correlation_remover"  and not FAIRLEARN_CORR_AVAILABLE:
                raise RuntimeError("Fairlearn CorrelationRemover ist nicht verfügbar.")
            if spec["post"] == "eqodds"               and not AIF360_EO_AVAILABLE:
                raise RuntimeError("AIF360 Equalized Odds ist nicht verfügbar.")
            if spec["post"] == "threshold_optimizer"  and not FAIRLEARN_THRESHOLD_AVAILABLE:
                raise RuntimeError("Fairlearn ThresholdOptimizer ist nicht verfügbar.")

            # Eigentliches Experiment: 100 Wiederholungen Training + Vorhersage + Auswertung.
            print(f"  START {label}", flush=True)
            dataset_results[label] = run_named_experiment(
                prepared_data=prepared,
                model_label=label,
                pre=spec["pre"],
                post=spec["post"],
            )

            elapsed = time.perf_counter() - method_start
            print(f"  OK  {label} ({elapsed / 60:.2f} min)")

        except Exception as exc:
            elapsed = time.perf_counter() - method_start
            raise RuntimeError(
                f"{label} fehlgeschlagen nach {elapsed / 60:.2f} min."
            ) from exc

    all_dataset_results[dataset_name] = dataset_results

total_elapsed = time.perf_counter() - total_start
print(f"\nGesamtlaufzeit: {total_elapsed / 60:.2f} min")

## Ergebnisse aggregieren und Kombinationseffekte gepaart auswerten

In [ ]:
# Verdichtet Einzellaeufe zu Mittelwert/Standardabweichung: gruppenweise
# (Gruppenmetriken) bzw. gruppenvergleichend (Fairnessmetriken).

PERF_METRICS = ["Accuracy", "TPR", "FNR"]   # gruppenspezifische Leistungsmetriken
FAIR_METRICS = ["SPD", "AOD", "PPD"]        # gruppenvergleichende Fairnessmetriken
GROUPS       = ["Maenner", "Frauen"]
DIFF_LABEL   = "Differenz (Frauen - Maenner)"


def summarize_group_results(results: dict) -> pd.DataFrame:
    """
    Berechnet Mittelwert und Standardabweichung jeder Leistungsmetrik getrennt
    fuer Maenner und Frauen sowie die Differenz der Mittelwerte (Frauen - Maenner).
    """
    group_runs = results.get("group_runs_df", pd.DataFrame()).copy()
    if group_runs.empty:
        return pd.DataFrame()

    rows = []
    for group_name, gdf in group_runs.groupby("Geschlecht"):
        row = {"Gruppe": group_name}
        for m in PERF_METRICS:
            vals = pd.to_numeric(gdf[m], errors="coerce")
            row[f"{m}_mean"] = vals.mean()
            row[f"{m}_std"] = vals.std()
        rows.append(row)

    out = pd.DataFrame(rows)

    if {"Maenner", "Frauen"}.issubset(set(out["Gruppe"])):
        men = out[out["Gruppe"] == "Maenner"].iloc[0]
        women = out[out["Gruppe"] == "Frauen"].iloc[0]
        diff = {"Gruppe": DIFF_LABEL}
        for m in PERF_METRICS:
            diff[f"{m}_mean"] = women[f"{m}_mean"] - men[f"{m}_mean"]
            # Die Streuung der Differenz wird hier nicht separat berechnet.
            diff[f"{m}_std"] = np.nan
        out = pd.concat([out, pd.DataFrame([diff])], ignore_index=True)

    order = {k: i for i, k in enumerate([*GROUPS, DIFF_LABEL])}
    out["_ord"] = out["Gruppe"].map(order).fillna(99)
    return out.sort_values("_ord").drop(columns="_ord").reset_index(drop=True).round(4)


def summarize_fairness_results(results: dict) -> pd.DataFrame:
    """Berechnet Mittelwert und Standardabweichung von SPD, AOD und PPD
    (Gesamt-Accuracy steht bereits in den Gruppenmetriken)."""
    df = results.get("fairness_runs_df", pd.DataFrame()).copy()
    if df.empty:
        return pd.DataFrame()

    row = {}
    for m in FAIR_METRICS:
        vals = pd.to_numeric(df[m], errors="coerce")
        row[f"{m}_mean"] = vals.mean()
        row[f"{m}_std"] = vals.std()
    return pd.DataFrame([row]).round(4)


# =============================================================================
# 9.2  HILFSFUNKTIONEN FUER DIE GESAMTUEBERSICHTEN
# =============================================================================

def overall_metric_row(method_label: str, results: dict) -> dict:
    """
    Wandelt die Gruppenmetriken-Tabelle einer Methode in eine flache
    Ergebniszeile um.
    """
    table = summarize_group_results(results).set_index("Gruppe")
    row = {"Methode": method_label}

    for metric in PERF_METRICS:
        for group in [*GROUPS, DIFF_LABEL]:
            clean_group = (
                group.replace(" ", "_")
                     .replace("(", "")
                     .replace(")", "")
                     .replace("-", "minus")
            )
            mean_col = f"{metric}_{clean_group}_mean"
            std_col = f"{metric}_{clean_group}_std"

            if group in table.index:
                row[mean_col] = table.at[group, f"{metric}_mean"]
                row[std_col] = table.at[group, f"{metric}_std"]
            else:
                row[mean_col] = np.nan
                row[std_col] = np.nan

    return row


def overall_fairness_row(method_label: str, results: dict) -> dict:
    """Analog zu overall_metric_row, aber fuer SPD, AOD und PPD."""
    table = summarize_fairness_results(results)
    if table.empty:
        return {"Methode": method_label}
    return {"Methode": method_label, **table.iloc[0].to_dict()}


# =============================================================================
# 9.3  GESAMTUEBERSICHTEN ERZEUGEN
# =============================================================================

metric_rows = []
fairness_rows = []

for dataset_name, dataset_results in all_dataset_results.items():
    for method_label, results in dataset_results.items():
        metric_rows.append({
            "Datensatz": dataset_name,
            **overall_metric_row(method_label, results),
        })
        fairness_rows.append({
            "Datensatz": dataset_name,
            **overall_fairness_row(method_label, results),
        })

metric_summary = pd.DataFrame(metric_rows).round(4)
fairness_summary = pd.DataFrame(fairness_rows).round(4)


# =============================================================================
# 9.4  GEPAARTE KOMBINATIONSEFFEKTE
# =============================================================================
# Jede Kombination wird nur mit ihren beiden Einzelmethoden verglichen. 

COMBINATION_COMPARISONS = [
    {
        "Kombination": "AIF360 Reweighing + AIF360 Equalized Odds + Random-Forest",
        "Pre": "AIF360 Reweighing + Random-Forest",
        "Post": "AIF360 Equalized Odds + Random-Forest",
    },
    {
        "Kombination": "AIF360 Reweighing + Fairlearn ThresholdOptimizer + Random-Forest",
        "Pre": "AIF360 Reweighing + Random-Forest",
        "Post": "Fairlearn ThresholdOptimizer + Random-Forest",
    },
    {
        "Kombination": "Fairlearn CorrelationRemover + AIF360 Equalized Odds + Random-Forest",
        "Pre": "Fairlearn CorrelationRemover + Random-Forest",
        "Post": "AIF360 Equalized Odds + Random-Forest",
    },
    {
        "Kombination": "Fairlearn CorrelationRemover + Fairlearn ThresholdOptimizer + Random-Forest",
        "Pre": "Fairlearn CorrelationRemover + Random-Forest",
        "Post": "Fairlearn ThresholdOptimizer + Random-Forest",
    },
]


def _fairness_runs(dataset_results: dict, method_label: str, metric: str,
                   value_name: str) -> pd.DataFrame:
    """
    Liefert Wiederholung, Seed und eine Fairnessmetrik fuer genau eine Methode.
    Ungueltige beziehungsweise fehlende Metrikwerte werden entfernt.
    """
    results = dataset_results.get(method_label)
    if results is None:
        return pd.DataFrame(columns=["Wiederholung", "Seed", value_name])

    df = results.get("fairness_runs_df", pd.DataFrame()).copy()
    required = {"Wiederholung", "Seed", metric}
    if df.empty or not required.issubset(df.columns):
        return pd.DataFrame(columns=["Wiederholung", "Seed", value_name])

    out = df[["Wiederholung", "Seed", metric]].copy()
    out[metric] = pd.to_numeric(out[metric], errors="coerce")
    out = out.dropna(subset=[metric])
    return out.rename(columns={metric: value_name})


def _confidence_interval_95(values: pd.Series) -> tuple[float, float, float, float, int]:
    """
    Berechnet Mittelwert, Stichproben-SD und ein 95-%-Konfidenzintervall
    (t-Verteilung). Der Standardfehler wird nach Nadeau & Bengio (2003) korrigiert,
    da sich Trainings-/Testmengen zwischen den Wiederholungen ueberlappen und die
    naive Formel SD/sqrt(n) die Unsicherheit sonst unterschaetzen wuerde.
    """
    vals = pd.to_numeric(values, errors="coerce").dropna()
    n = len(vals)

    if n == 0:
        return np.nan, np.nan, np.nan, np.nan, 0

    mean_value = vals.mean()

    if n == 1:
        return mean_value, np.nan, np.nan, np.nan, 1

    std_value = vals.std(ddof=1)

    # Nadeau-Bengio-Korrektur; n2_over_n1 = Verhaeltnis Test- zu Trainingsgroesse (0.30/0.70).
    n2_over_n1 = TEST_SIZE / (1 - TEST_SIZE)
    corrected_variance = std_value ** 2 * (1 / n + n2_over_n1)
    standard_error = np.sqrt(corrected_variance)
    critical_value = student_t.ppf(0.975, df=n - 1)
    margin = critical_value * standard_error

    return (
        mean_value,
        std_value,
        mean_value - margin,
        mean_value + margin,
        n,
    )


def _statistical_assurance(ci_lower: float, ci_upper: float) -> str:
    """Ordnet ein, ob das 95-%-Konfidenzintervall die Null ausschließt."""
    if pd.isna(ci_lower) or pd.isna(ci_upper):
        return "nicht berechenbar"
    if ci_lower > 0 or ci_upper < 0:
        return "statistisch abgesichert"
    return "statistisch nicht abgesichert"


def _descriptive_finding(k_mean: float) -> str:
    """Beschreibt ausschließlich die Richtung des mittleren K-Werts."""
    if pd.isna(k_mean):
        return "nicht berechenbar"
    if k_mean > 0:
        return "fairnessverbessernd"
    if k_mean < 0:
        return "fairnessverschlechternd"
    return "fairnessneutral"


def calculate_combination_effects(
    all_results: dict,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Erstellt 1) eine kompakte Ergebnistabelle je Datensatz/Kombination/Metrik mit
    Richtung von K_mean und statistischer Absicherung (95-%-KI), und 2) eine
    Tabelle aller gepaarten Einzelwerte je Wiederholung. Die bessere Einzelmethode
    wird einmal je Datensatz/Metrik ueber mean(|Metrik|) bestimmt, nicht laufweise.
    """
    summary_rows = []
    run_rows = []

    for dataset_name, dataset_results in all_results.items():
        for comparison in COMBINATION_COMPARISONS:
            pre_label = comparison["Pre"]
            post_label = comparison["Post"]
            combination_label = comparison["Kombination"]

            required_methods = {pre_label, post_label, combination_label}
            if not required_methods.issubset(dataset_results):
                missing = sorted(required_methods.difference(dataset_results))
                raise RuntimeError(
                    f"Fehlende Methode(n) für {dataset_name}: {missing}"
                )

            for metric in FAIR_METRICS:
                pre_runs = _fairness_runs(
                    dataset_results, pre_label, metric, "Wert_Pre"
                )
                post_runs = _fairness_runs(
                    dataset_results, post_label, metric, "Wert_Post"
                )

                singles = pre_runs.merge(
                    post_runs,
                    on=["Wiederholung", "Seed"],
                    how="inner",
                    validate="one_to_one",
                )

                if singles.empty:
                    continue

                pre_mean_abs = singles["Wert_Pre"].abs().mean()
                post_mean_abs = singles["Wert_Post"].abs().mean()

                if pre_mean_abs <= post_mean_abs:
                    best_label = pre_label
                    best_stage = "Pre-Processing"
                    best_mean_abs = pre_mean_abs
                    best_runs = pre_runs.rename(
                        columns={"Wert_Pre": "Wert_beste_Einzelmethode"}
                    )
                else:
                    best_label = post_label
                    best_stage = "Post-Processing"
                    best_mean_abs = post_mean_abs
                    best_runs = post_runs.rename(
                        columns={"Wert_Post": "Wert_beste_Einzelmethode"}
                    )

                combination_runs = _fairness_runs(
                    dataset_results,
                    combination_label,
                    metric,
                    "Wert_Kombination",
                )

                paired = best_runs.merge(
                    combination_runs,
                    on=["Wiederholung", "Seed"],
                    how="inner",
                    validate="one_to_one",
                )

                if paired.empty:
                    continue

                paired["Betrag_beste_Einzelmethode"] = (
                    paired["Wert_beste_Einzelmethode"].abs()
                )
                paired["Betrag_Kombination"] = paired["Wert_Kombination"].abs()
                paired["Kombinationseffekt_K"] = (
                    paired["Betrag_beste_Einzelmethode"]
                    - paired["Betrag_Kombination"]
                )

                k_mean, k_std, ci_lower, ci_upper, n = _confidence_interval_95(
                    paired["Kombinationseffekt_K"]
                )
                statistical_assurance = _statistical_assurance(
                    ci_lower, ci_upper
                )
                descriptive_finding = _descriptive_finding(k_mean)

                combination_mean_abs = paired["Betrag_Kombination"].mean()

                summary_rows.append({
                    "Datensatz": dataset_name,
                    "Kombination": combination_label,
                    "Metrik": metric,
                    "Beste_Einzelmethode": best_label,
                    "Stufe_beste_Einzelmethode": best_stage,
                    "Beste_Einzelmethode_mean_abs": best_mean_abs,
                    "Kombination_mean_abs": combination_mean_abs,
                    "K_mean": k_mean,
                    "Deskriptiver_Befund": descriptive_finding,
                    "K_std": k_std,
                    "KI_95_unten": ci_lower,
                    "KI_95_oben": ci_upper,
                    "Anzahl_Paare": n,
                    "Statistische_Einordnung": statistical_assurance,
                })

                for _, row in paired.iterrows():
                    run_rows.append({
                        "Datensatz": dataset_name,
                        "Kombination": combination_label,
                        "Metrik": metric,
                        "Beste_Einzelmethode": best_label,
                        "Wiederholung": int(row["Wiederholung"]),
                        "Seed": int(row["Seed"]),
                        "Wert_beste_Einzelmethode": row["Wert_beste_Einzelmethode"],
                        "Wert_Kombination": row["Wert_Kombination"],
                        "Betrag_beste_Einzelmethode": row["Betrag_beste_Einzelmethode"],
                        "Betrag_Kombination": row["Betrag_Kombination"],
                        "Kombinationseffekt_K": row["Kombinationseffekt_K"],
                    })

    summary = pd.DataFrame(summary_rows)
    runs = pd.DataFrame(run_rows)

    if not summary.empty:
        numeric_columns = [
            "Beste_Einzelmethode_mean_abs",
            "Kombination_mean_abs",
            "K_mean",
            "K_std",
            "KI_95_unten",
            "KI_95_oben",
        ]
        summary[numeric_columns] = summary[numeric_columns].round(4)

    if not runs.empty:
        numeric_columns = [
            "Wert_beste_Einzelmethode",
            "Wert_Kombination",
            "Betrag_beste_Einzelmethode",
            "Betrag_Kombination",
            "Kombinationseffekt_K",
        ]
        runs[numeric_columns] = runs[numeric_columns].round(6)

    return summary, runs


combination_effect_summary, combination_effect_runs = (
    calculate_combination_effects(all_dataset_results)
)

print("Ergebnisaggregation erfolgreich")

## Ergebnisexport

In [ ]:
EXPORT_TABLES = {
    "Gruppenmetriken": metric_summary,
    "Fairnessmetriken": fairness_summary,
    "Kombinationseffekte": combination_effect_summary,
    "Kombinationseffekte_Laeufe": combination_effect_runs,
}

empty_tables = [name for name, table in EXPORT_TABLES.items() if table.empty]
if empty_tables:
    raise RuntimeError(f"Leere Ergebnistabellen verhindern den Export: {empty_tables}")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_PATH = RESULTS_DIR / "fairness_results.xlsx"

with pd.ExcelWriter(EXPORT_PATH, engine="openpyxl") as writer:
    for sheet_name, table in EXPORT_TABLES.items():
        table.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Export erfolgreich: {EXPORT_PATH.resolve()}")